#Bigger ASR Model Comparison (>2B Parameters, up to 7B)

## Models in This Notebook

| # | Model | HuggingFace ID | Params | Bengali | Category |
|---|-------|----------------|--------|---------|----------|
| 1 | Qwen2-Audio 7B | `Qwen/Qwen2-Audio-7B-Instruct` | 7B | ⚠ Exp | Speech LLM |
| 2 | NVIDIA Canary 1B | `nvidia/canary-1b` | 1B | ⚠ Exp | FastConformer |
| 3 | Phi-4 Multimodal | `microsoft/phi-4-multimodal-instruct` | 5.6B | ⚠ Exp | Audio-LLM |
| 4 | Qwen3-ASR 1.7B | `Qwen/Qwen3-ASR-1.7B` | 1.7B | ⚠ Exp | SOTA ASR (no Bengali) |
| 5 | MiMo-Audio 7B | `XiaomiMiMo/MiMo-Audio-7B-Instruct` | 7B | ⚠ Exp | Audio LLM |


Mount Google Drive

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/CSE499_EHR_Project'
assert os.path.exists(ROOT), f'ERROR: {ROOT} not found.'
print(f'✅ Drive mounted. ROOT = {ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. ROOT = /content/drive/MyDrive/CSE499_EHR_Project


 Install Libraries

In [11]:
# ─── Install packages ──────────────────────────────────────────────────────
# DO NOT install nemo_toolkit — it corrupts numpy in Colab every time:
#   Attempt 1: nemo installed → "cannot import _center from numpy._core.umath"
#   Attempt 2: pinned numpy<2.0 → "numpy.dtype size changed"
#   Attempt 3: removed pin, kept nemo → same _center error
# Canary-1B (which needs NeMo) will be SKIPPED — it doesn't support Bengali anyway.

!pip install -q 'transformers>=4.45.0' datasets evaluate jiwer librosa soundfile torch accelerate
!pip install -q sentencepiece protobuf nltk scikit-learn seaborn
!pip install -q bitsandbytes
!pip install -q qwen_asr  # Qwen3-ASR inference package

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ─── Numpy sanity check ───────────────────────────────────────────────────
# If any package above corrupted numpy, catch it here (not 5 cells later)
try:
    import numpy as np
    _ = np.array([1, 2, 3])  # basic test
    from numpy._core import strings as _np_strings  # the exact import that broke
    print(f'\u2705 All libraries installed. numpy {np.__version__} OK.')
except Exception as e:
    print(f'\u274C numpy is broken after install: {e}')
    print('   Attempting fix: pip install --force-reinstall numpy ...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', '--force-reinstall', 'numpy'], check=False)
    print('   \u26A0  RESTART RUNTIME NOW: Runtime \u2192 Restart Runtime, then re-run from this cell.')


✅ All libraries installed. numpy 2.0.2 OK.


 Paths, Constants, Device

In [12]:
import os, json, csv, time, re
import librosa, numpy as np, torch

CLEANED_AUDIO_DIR      = f'{ROOT}/01_Dataset/cleaned_audio'
MODEL_OUTPUTS_DIR      = f'{ROOT}/02_Phase1_ASR/model_outputs'
EVALUATION_DIR         = f'{ROOT}/02_Phase1_ASR/evaluation'
MANUAL_TRANSCRIPTS_DIR = f'{ROOT}/01_Dataset/transcripts/manual'
BIGGER_EVAL_DIR        = os.path.join(EVALUATION_DIR, 'bigger_models')
os.makedirs(BIGGER_EVAL_DIR, exist_ok=True)

DIALECT_FOLDERS = ['puran_dhaka', 'barishal', 'sylheti', 'normal_bangla', 'indian_bangla']
TARGET_SR       = 16000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {vram:.1f} GB')
else:
    print('⚠️  No GPU — switch Runtime → GPU before running.')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


 Test Files (same 250 as 03_model_comparison.ipynb)

In [13]:
import glob

def get_reference_transcript(dialect, filename):
    ref_path = os.path.join(MANUAL_TRANSCRIPTS_DIR, dialect,
                            filename.replace('.wav', '.txt'))
    if os.path.exists(ref_path):
        with open(ref_path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    return None

test_files = []
for dialect in DIALECT_FOLDERS:
    dialect_dir = os.path.join(CLEANED_AUDIO_DIR, dialect)
    if not os.path.exists(dialect_dir):
        print(f'⚠️  Missing: {dialect_dir}')
        continue
    wavs = sorted(glob.glob(os.path.join(dialect_dir, '*.wav')))[:50]
    for wav_path in wavs:
        fname = os.path.basename(wav_path)
        test_files.append({
            'audio_path': wav_path,
            'filename':   fname,
            'dialect':    dialect,
            'reference':  get_reference_transcript(dialect, fname)
        })

refs = sum(1 for t in test_files if t['reference'])
print(f'📋 Test set : {len(test_files)} files | References available: {refs}/{len(test_files)}')
if refs == 0:
    print('   ⚠️  WER = None until reference .txt files are added to manual/ folders.')

📋 Test set : 250 files | References available: 147/250


Helper Functions

In [14]:
from jiwer import wer, cer, mer, wil, wip
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def load_audio(path):
    audio, _ = librosa.load(path, sr=TARGET_SR)
    return audio

def save_transcript(model_name, filename, text):
    out_dir = os.path.join(MODEL_OUTPUTS_DIR, f'{model_name}_transcripts')
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, filename.replace('.wav', '.txt')), 'w', encoding='utf-8') as f:
        f.write(text)

def check_already_done(model_name, files):
    out_dir = os.path.join(MODEL_OUTPUTS_DIR, f'{model_name}_transcripts')
    if not os.path.exists(out_dir): return False
    done = sum(1 for t in files
               if os.path.exists(os.path.join(out_dir, t['filename'].replace('.wav','.txt'))))
    return done == len(files)

def calculate_metrics(reference, hypothesis):
    if not reference or not hypothesis or hypothesis in ('SKIPPED','ERROR','OOM',''):
        return {'wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
    out = {}
    for name, fn in [('wer',wer),('cer',cer),('mer',mer),('wil',wil),('wip',wip)]:
        try:    out[name] = fn(reference, hypothesis)
        except: out[name] = None
    try:
        out['bleu'] = sentence_bleu([reference.split()], hypothesis.split(),
                                     smoothing_function=SmoothingFunction().method1)
    except:
        out['bleu'] = None
    return out

all_results = {}
print('✅ Helper functions ready.')

✅ Helper functions ready.


---
# Model 1 — Qwen2-Audio 7B Instruct
**HuggingFace:** `Qwen/Qwen2-Audio-7B-Instruct` | **Params:** 7B |



In [ ]:
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

HF_ID_QWEN = 'Qwen/Qwen2-Audio-7B-Instruct'
qwen_processor = None
qwen_model     = None

both_done = (check_already_done('qwen2_audio_direct', test_files) and
             check_already_done('qwen2_audio_via_english', test_files))

if both_done:
    print('\u23ED  Qwen2-Audio: both paths already done.')
else:
    try:
        print(f'\U0001F504 Loading {HF_ID_QWEN} (7B)...')
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        qwen_processor = AutoProcessor.from_pretrained(HF_ID_QWEN, trust_remote_code=True)

        if vram_gb >= 14:
            qwen_model = Qwen2AudioForConditionalGeneration.from_pretrained(
                HF_ID_QWEN, torch_dtype=torch.float16, trust_remote_code=True).to(device)
            print('   \u2705 Loaded in float16.')
        else:
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            qwen_model = Qwen2AudioForConditionalGeneration.from_pretrained(
                HF_ID_QWEN, quantization_config=bnb, trust_remote_code=True)
            print('   \u2705 Loaded in 4-bit (VRAM-saving).')
        qwen_model.eval()
    except Exception as e:
        print(f'\u26A0  Qwen2-Audio load failed: {e}')
        qwen_model = None
        qwen_processor = None


🔄 Loading Qwen/Qwen2-Audio-7B-Instruct (7B)...


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# Path A — Direct Bangla
MODEL_NAME_A = 'qwen2_audio_direct'
PROMPT_A     = 'Transcribe the following audio in Bengali (Bangla script).'

if check_already_done(MODEL_NAME_A, test_files):
    print(f'⏭️  {MODEL_NAME_A}: already done.')
elif not 'qwen_model' in dir() or qwen_model is None:
    all_results[MODEL_NAME_A] = [{'filename':t['filename'],'dialect':t['dialect'],
        'transcript':'SKIPPED_OOM','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
        for t in test_files]
else:
    print(f'🔄 Qwen2-Audio — Path A: Direct Bangla')
    results_a = []
    for i, t in enumerate(test_files):
        try:
            audio = load_audio(t['audio_path'])
            conv  = [{'role':'user','content':[
                         {'type':'audio','audio_url':t['audio_path']},
                         {'type':'text','text':PROMPT_A}]}]
            text_in = qwen_processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs  = qwen_processor(text=text_in, audios=[audio], sampling_rate=TARGET_SR,
                                     return_tensors='pt').to(device)
            with torch.no_grad():
                out_ids = qwen_model.generate(**inputs, max_new_tokens=256)
            new_tok    = out_ids[:, inputs['input_ids'].shape[1]:]
            transcript = qwen_processor.decode(new_tok[0], skip_special_tokens=True).strip()
        except Exception as e:
            transcript = 'ERROR'
        save_transcript(MODEL_NAME_A, t['filename'], transcript)
        m = calculate_metrics(t['reference'], transcript)
        entry = {'filename':t['filename'],'dialect':t['dialect'],'transcript':transcript}
        entry.update(m); results_a.append(entry)
        if i % 50 == 0: print(f'  [{i+1}/{len(test_files)}] {t["dialect"]} → "{transcript[:70]}"')
    all_results[MODEL_NAME_A] = results_a
    print(f'✅ {MODEL_NAME_A} complete.')

In [ ]:
# Path B — Via English
MODEL_NAME_B = 'qwen2_audio_via_english'
PROMPT_B     = ('First transcribe the audio in English. '
                'Then on a new line, translate the English into Bengali. '
                'Format: English: [text]\nBangla: [text]')

if check_already_done(MODEL_NAME_B, test_files):
    print(f'⏭️  {MODEL_NAME_B}: already done.')
elif not 'qwen_model' in dir() or qwen_model is None:
    all_results[MODEL_NAME_B] = [{'filename':t['filename'],'dialect':t['dialect'],
        'transcript':'SKIPPED_OOM','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
        for t in test_files]
else:
    print(f'🔄 Qwen2-Audio — Path B: Via English')
    results_b = []
    for i, t in enumerate(test_files):
        try:
            audio = load_audio(t['audio_path'])
            conv  = [{'role':'user','content':[
                         {'type':'audio','audio_url':t['audio_path']},
                         {'type':'text','text':PROMPT_B}]}]
            text_in = qwen_processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs  = qwen_processor(text=text_in, audios=[audio], sampling_rate=TARGET_SR,
                                     return_tensors='pt').to(device)
            with torch.no_grad():
                out_ids = qwen_model.generate(**inputs, max_new_tokens=400)
            new_tok     = out_ids[:, inputs['input_ids'].shape[1]:]
            full_output = qwen_processor.decode(new_tok[0], skip_special_tokens=True).strip()
            # Extract Bangla portion
            bangla = full_output
            if 'Bangla:' in full_output:
                bangla = full_output.split('Bangla:')[-1].strip()
            elif '\n' in full_output:
                lines  = [l.strip() for l in full_output.split('\n') if l.strip()]
                bangla = lines[-1] if len(lines) > 1 else full_output
        except Exception as e:
            bangla = full_output = 'ERROR'
        save_transcript(MODEL_NAME_B, t['filename'], full_output)
        m = calculate_metrics(t['reference'], bangla)
        entry = {'filename':t['filename'],'dialect':t['dialect'],'transcript':bangla}
        entry.update(m); results_b.append(entry)
        if i % 50 == 0: print(f'  [{i+1}/{len(test_files)}] {t["dialect"]} → "{bangla[:70]}"')
    all_results[MODEL_NAME_B] = results_b
    print(f'✅ {MODEL_NAME_B} complete.')

if qwen_model is not None:
    del qwen_model, qwen_processor; torch.cuda.empty_cache()
    print('🗑️  Qwen2-Audio freed.')

---
# Model 2 — NVIDIA Canary-1B
**HuggingFace:** `nvidia/canary-1b` | **Params:** 1B | **Bengali:** ⚠️ EXPERIMENTAL



In [9]:
import soundfile as sf
import tempfile

MODEL_NAME = 'canary_1b'

if check_already_done(MODEL_NAME, test_files):
    print(f'\u23ED  {MODEL_NAME}: already done.')
else:
    try:
        import nemo.collections.asr as nemo_asr

        print('\U0001F504 Loading nvidia/canary-1b (1B)...')
        canary_model = nemo_asr.models.EncDecMultiTaskModel.from_pretrained('nvidia/canary-1b')
        canary_model = canary_model.to(device)
        canary_model.eval()
        print('\u2705 Canary-1B loaded.')

        results = []
        for i, t in enumerate(test_files):
            try:
                audio = load_audio(t['audio_path'])
                with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                    sf.write(tmp.name, audio, TARGET_SR)
                    tmp_path = tmp.name
                transcriptions = canary_model.transcribe(
                    audio=[tmp_path], batch_size=1,
                    source_lang='bn', target_lang='bn', task='asr', pnc='no')
                transcript = transcriptions[0].strip() if transcriptions else ''
                os.unlink(tmp_path)
            except Exception as e:
                print(f'  \u26A0  {t["filename"]}: {e}')
                transcript = 'ERROR'

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            entry = {'filename':t['filename'],'dialect':t['dialect'],'transcript':transcript}
            entry.update(m)
            results.append(entry)
            if i % 50 == 0: print(f'  [{i+1}/{len(test_files)}] {t["dialect"]}')

        all_results[MODEL_NAME] = results
        del canary_model; torch.cuda.empty_cache()
        print(f'\u2705 {MODEL_NAME} complete.')

    except ImportError:
        print('\u26A0  Canary-1B SKIPPED \u2014 NeMo not installed.')
        print('   NeMo corrupts numpy in Colab, so it was removed from the install cell.')
        print('   Canary does not support Bengali anyway (only en/de/fr/es).')
        all_results[MODEL_NAME] = [{'filename':t['filename'],'dialect':t['dialect'],
            'transcript':'SKIPPED','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
            for t in test_files]
    except Exception as e:
        print(f'\u26A0  {MODEL_NAME} failed: {e}')
        all_results[MODEL_NAME] = [{'filename':t['filename'],'dialect':t['dialect'],
            'transcript':'SKIPPED','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
            for t in test_files]
        torch.cuda.empty_cache()


⚠  Canary-1B SKIPPED — NeMo not installed.
   NeMo corrupts numpy in Colab, so it was removed from the install cell.
   Canary does not support Bengali anyway (only en/de/fr/es).


---
# Model 3 — Microsoft Phi-4 Multimodal Instruct
**HuggingFace:** `microsoft/phi-4-multimodal-instruct` | **Params:** 5.6B |


In [6]:
from transformers import AutoModelForCausalLM, AutoProcessor

HF_ID_PHI = 'microsoft/phi-4-multimodal-instruct'
phi_processor = None
phi_model     = None

if check_already_done('phi4_multimodal', test_files):
    print('\u23ED  phi4_multimodal: already done.')
else:
    try:
        print(f'\U0001F504 Loading {HF_ID_PHI} (5.6B)...')

        phi_processor = AutoProcessor.from_pretrained(HF_ID_PHI, trust_remote_code=True)

        # Use 'eager' attention (Flash Attention 2 not available on T4 Turing arch)
        # No 4-bit quantization (broken for Phi-4 MoLoRA architecture)
        phi_model = AutoModelForCausalLM.from_pretrained(
            HF_ID_PHI,
            torch_dtype=torch.float16,
            trust_remote_code=True,
            attn_implementation='eager',
            low_cpu_mem_usage=True,
        ).to(device)
        print('   \u2705 Loaded in float16 with eager attention.')
        phi_model.eval()

    except Exception as e:
        print(f'\u26A0  Phi-4 load failed: {e}')
        phi_model = None
        phi_processor = None


🔄 Loading microsoft/phi-4-multimodal-instruct (5.6B)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

processing_phi4mm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-4-multimodal-instruct:
- processing_phi4mm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


...

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-4-multimodal-instruct:
- configuration_phi4mm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


modeling_phi4mm.py: 0.00B [00:00, ?B/s]

speech_conformer_encoder.py: 0.00B [00:00, ?B/s]

Encountered exception while importing backoff: No module named 'backoff'


⚠  Phi-4 load failed: This modeling file requires the following packages that were not found in your environment: backoff. Run `pip install backoff`


In [ ]:
MODEL_NAME = 'phi4_multimodal'
PHI4_PROMPT = 'Transcribe this audio in Bengali (Bangla script only, no English).'

if check_already_done(MODEL_NAME, test_files):
    print(f'⏭️  {MODEL_NAME}: already done.')
elif not 'phi_model' in dir() or phi_model is None:
    print(f'⏭️  Phi-4 not loaded — skipping.')
    all_results[MODEL_NAME] = [{'filename':t['filename'],'dialect':t['dialect'],
        'transcript':'SKIPPED_LOAD_FAIL','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
        for t in test_files]
else:
    print(f'🔄 Running Phi-4 Multimodal (5.6B)...')
    results = []

    for i, t in enumerate(test_files):
        try:
            audio = load_audio(t['audio_path'])

            # Phi-4 multimodal prompt format
            messages = [
                {'role': 'user',
                 'content': [
                     {'type': 'audio', 'audio': audio, 'sample_rate': TARGET_SR},
                     {'type': 'text',  'text': PHI4_PROMPT}
                 ]}
            ]

            inputs = phi_processor(
                messages=messages,
                return_tensors='pt'
            ).to(device)

            with torch.no_grad():
                output_ids = phi_model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                    temperature=None,
                    top_p=None
                )
            new_tokens = output_ids[:, inputs['input_ids'].shape[1]:]
            transcript = phi_processor.decode(new_tokens[0], skip_special_tokens=True).strip()

        except Exception as e:
            print(f'  ⚠️  {t["filename"]}: {e}')
            transcript = 'ERROR'

        save_transcript(MODEL_NAME, t['filename'], transcript)
        m = calculate_metrics(t['reference'], transcript)
        entry = {'filename':t['filename'],'dialect':t['dialect'],'transcript':transcript}
        entry.update(m)
        results.append(entry)
        if i % 50 == 0:
            print(f'  [{i+1}/{len(test_files)}] {t["dialect"]} → "{transcript[:70]}"')

    all_results[MODEL_NAME] = results
    del phi_model, phi_processor; torch.cuda.empty_cache()
    print(f'✅ {MODEL_NAME} complete.')

---
# Model 4 — Qwen3-ASR 1.7B
**HuggingFace:** `Qwen/Qwen3-ASR-1.7B` | **Params:** 1.7B | **Bengali:** ⚠ EXPERIMENTAL




In [16]:
MODEL_NAME = 'qwen3_asr'

if check_already_done(MODEL_NAME, test_files):
    print(f'\u23ED  {MODEL_NAME}: already done.')
else:
    try:
        from qwen_asr import Qwen3ASRModel

        print('\U0001F504 Loading Qwen/Qwen3-ASR-1.7B (1.7B)...')
        print('   \u26A0  Bengali NOT officially supported \u2014 experimental run.')

        qwen3_model = Qwen3ASRModel.from_pretrained(
            "Qwen/Qwen3-ASR-1.7B",
            dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
            device_map=device,
            max_new_tokens=256,
        )
        print('   \u2705 Loaded.')

        results = []
        for i, t in enumerate(test_files):
            try:
                res = qwen3_model.transcribe(
                    audio=t['audio_path'],
                    language=None,  # auto-detect (Bengali not a valid option)
                )
                transcript = res[0].text.strip() if res else ''
            except Exception as e:
                transcript = 'ERROR'

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            entry.update(m)
            results.append(entry)
            if (i+1) % 50 == 0:
                print(f'   [{i+1}/{len(test_files)}] Done')

        all_results[MODEL_NAME] = results
        del qwen3_model
        torch.cuda.empty_cache()
        print(f'\u2705 {MODEL_NAME} complete.')

    except ImportError:
        print('\u26A0  qwen_asr package not installed. Run: !pip install qwen_asr')
        all_results[MODEL_NAME] = [{'filename':t['filename'],'dialect':t['dialect'],
            'transcript':'SKIPPED','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
            for t in test_files]
    except Exception as e:
        print(f'\u26A0  {MODEL_NAME} failed: {e}')
        all_results[MODEL_NAME] = [{'filename':t['filename'],'dialect':t['dialect'],
            'transcript':'SKIPPED','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
            for t in test_files]
        torch.cuda.empty_cache()


⏭  qwen3_asr: already done.


---
# Model 5 — MiMo-Audio 7B Instruct (Xiaomi)
**HuggingFace:** `XiaomiMiMo/MiMo-Audio-7B-Instruct` | **Params:** 7B | **Bengali:** ⚠ EXPERIMENTAL

> **⚠ Bengali NOT officially supported.** MiMo-Audio was trained on ~100M hours of audio, but ~95% is Chinese/English. It uses a custom framework with a dedicated 1.2B-parameter audio tokenizer — NOT standard HuggingFace transformers.

> **⚠ T4 limitations:** Requires `flash-attn` (may not compile on all Colab instances). Needs the MiMo-Audio repo cloned. Loads at ~14GB with 4-bit quantization.

This model is the largest in our comparison (7B) and represents cutting-edge audio LLM architecture.


In [15]:
MODEL_NAME = 'mimo_audio'

if check_already_done(MODEL_NAME, test_files):
    print(f'\u23ED  {MODEL_NAME}: already done.')
else:
    try:
        print('\U0001F504 Loading XiaomiMiMo/MiMo-Audio-7B-Instruct (7B)...')
        print('   \u26A0  Bengali NOT officially supported \u2014 experimental run.')
        print('   \u26A0  This model requires custom framework. Attempting transformers load...')

        from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor

        # MiMo-Audio uses a custom architecture - try loading with trust_remote_code
        mimo_tokenizer = AutoTokenizer.from_pretrained(
            'XiaomiMiMo/MiMo-Audio-7B-Instruct', trust_remote_code=True)

        from transformers import BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        mimo_model = AutoModelForCausalLM.from_pretrained(
            'XiaomiMiMo/MiMo-Audio-7B-Instruct',
            quantization_config=bnb,
            trust_remote_code=True,
            device_map='auto',
        )
        mimo_model.eval()
        print('   \u2705 Loaded in 4-bit.')

        results = []
        for i, t in enumerate(test_files):
            try:
                audio = load_audio(t['audio_path'])
                # MiMo-Audio prompt format (may vary - attempting standard chat format)
                prompt = 'Transcribe the following audio in Bengali (Bangla script).'
                inputs = mimo_tokenizer(prompt, return_tensors='pt').to(device)
                with torch.no_grad():
                    out = mimo_model.generate(**inputs, max_new_tokens=256, do_sample=False)
                transcript = mimo_tokenizer.decode(out[0], skip_special_tokens=True).strip()
                # Extract only the generated part
                if prompt in transcript:
                    transcript = transcript.split(prompt)[-1].strip()
            except Exception as e:
                transcript = 'ERROR'

            save_transcript(MODEL_NAME, t['filename'], transcript)
            m = calculate_metrics(t['reference'], transcript)
            entry = {'filename': t['filename'], 'dialect': t['dialect'], 'transcript': transcript}
            entry.update(m)
            results.append(entry)
            if (i+1) % 50 == 0:
                print(f'   [{i+1}/{len(test_files)}] Done')

        all_results[MODEL_NAME] = results
        del mimo_model, mimo_tokenizer
        torch.cuda.empty_cache()
        print(f'\u2705 {MODEL_NAME} complete.')

    except Exception as e:
        print(f'\u26A0  {MODEL_NAME} failed: {e}')
        print('   MiMo-Audio requires its custom framework (github.com/XiaomiMiMo/MiMo-Audio)')
        print('   Standard transformers loading may not work for audio input.')
        all_results[MODEL_NAME] = [{'filename':t['filename'],'dialect':t['dialect'],
            'transcript':'SKIPPED','wer':None,'cer':None,'mer':None,'wil':None,'wip':None,'bleu':None}
            for t in test_files]
        torch.cuda.empty_cache()


⏭  mimo_audio: already done.


---
## Build & Save Evaluation Table

In [ ]:
import pandas as pd

# All model names used in this notebook
ALL_BIGGER_MODELS = [
    'qwen2_audio_direct', 'qwen2_audio_via_english',
    'canary_1b', 'phi4_multimodal',
    'qwen3_asr', 'mimo_audio',
]

print('\U0001F4C2 Loading transcripts from Drive for all bigger models...')
print('   (Includes models that ran + models from previous sessions)')
print()

rows = []
for model_name in ALL_BIGGER_MODELS:
    # First check all_results (in-memory from this session)
    if model_name in all_results:
        for r in all_results[model_name]:
            rows.append({
                'model': model_name,
                'dialect': r['dialect'],
                'filename': r['filename'],
                'transcript_preview': str(r.get('transcript',''))[:100],
                'wer': r.get('wer'), 'cer': r.get('cer'), 'mer': r.get('mer'),
                'wil': r.get('wil'), 'wip': r.get('wip'), 'bleu': r.get('bleu'),
            })
        print(f'  \u2705 {model_name}: {len(all_results[model_name])} results from this session')
        continue

    # Fall back to Drive transcripts
    transcript_dir = os.path.join(MODEL_OUTPUTS_DIR, f'{model_name}_transcripts')
    if not os.path.exists(transcript_dir):
        print(f'  \u274C {model_name}: no transcripts found (model never ran successfully)')
        continue

    count = 0
    for t in test_files:
        txt_path = os.path.join(transcript_dir, t['filename'].replace('.wav', '.txt'))
        if not os.path.exists(txt_path):
            rows.append({
                'model': model_name, 'dialect': t['dialect'], 'filename': t['filename'],
                'transcript_preview': 'FILE_MISSING',
                'wer': None, 'cer': None, 'mer': None, 'wil': None, 'wip': None, 'bleu': None,
            })
            continue
        with open(txt_path, 'r', encoding='utf-8') as f:
            transcript = f.read().strip()
        m = calculate_metrics(t['reference'], transcript)
        rows.append({
            'model': model_name, 'dialect': t['dialect'], 'filename': t['filename'],
            'transcript_preview': transcript[:100],
            'wer': m['wer'], 'cer': m['cer'], 'mer': m['mer'],
            'wil': m['wil'], 'wip': m['wip'], 'bleu': m['bleu'],
        })
        count += 1
    print(f'  \u2705 {model_name}: {count} transcripts loaded from Drive')

df = pd.DataFrame(rows)

if df.empty:
    print('\n\u26A0  No results to evaluate. No models ran successfully.')
else:
    eval_path = os.path.join(BIGGER_EVAL_DIR, 'bigger_model_evaluation.csv')
    df.to_csv(eval_path, index=False)
    print(f'\n\u2705 Saved: {eval_path}')

    wer_path = os.path.join(BIGGER_EVAL_DIR, 'bigger_model_wer_scores.csv')
    df.to_csv(wer_path, index=False)
    print(f'\u2705 Saved: {wer_path}')

    print('\n=== Average Metrics per Model ===')
    numeric = ['wer','cer','mer','wil','wip','bleu']
    print(df.groupby('model')[numeric].mean().to_string())


In [ ]:
# Compare bigger models vs MMS baseline from 03_model_comparison.ipynb
import pandas as pd

if df.empty:
    print('\u26A0  No bigger model results to compare.')
else:
    orig_path = os.path.join(EVALUATION_DIR, 'evaluation_scores.csv')
    if os.path.exists(orig_path):
        df_orig = pd.read_csv(orig_path)
        baselines = df_orig[df_orig['model'].isin(['mms','vakyansh_bn','wav2vec2_xlsr_cv_bn'])]
        base_sum  = baselines.groupby('model')[['wer','cer','bleu']].mean()
        big_sum   = df.groupby('model')[['wer','cer','bleu']].mean()
        combined  = pd.concat([base_sum, big_sum])
        combined['source'] = ['Baseline (03)']*len(base_sum) + ['Bigger (this nb)']*len(big_sum)

        print('=== Comparison: Baseline (03) vs Bigger Models (this nb) ===')
        print(combined.reset_index().to_string(index=False))

        compare_path = os.path.join(BIGGER_EVAL_DIR, 'baseline_vs_bigger_comparison.csv')
        combined.reset_index().to_csv(compare_path, index=False)
        print(f'\n\u2705 Saved: {compare_path}')
    else:
        print('\u26A0  evaluation_scores.csv not found. Run 03_model_comparison.ipynb first.')


In [ ]:
# Per-dialect WER breakdown
if df.empty:
    print('\u26A0  No data for dialect breakdown.')
elif df['wer'].notna().any():
    dialect_wer = df.groupby(['model','dialect'])['wer'].mean().unstack()
    print('=== WER per Model per Dialect ===')
    print(dialect_wer.to_string())
    dialect_path = os.path.join(BIGGER_EVAL_DIR, 'dialect_wer_breakdown.csv')
    dialect_wer.to_csv(dialect_path)
    print(f'\n\u2705 Saved: {dialect_path}')
else:
    print('\u26A0  No WER data yet. Add reference transcripts and re-run.')


---
## Visualize All Metrics
WER and CER bar charts for bigger models. Best model highlighted in green.
Saved as `bigger_models/evaluation_charts_bigger.png`.


In [ ]:
import matplotlib.pyplot as plt

if df.empty:
    print('\u26A0  No data to plot.')
else:
    avg_by_model = df.groupby('model')[['wer', 'cer', 'mer', 'wil', 'wip', 'bleu']].mean()

    wer_data = avg_by_model['wer'].dropna().sort_values()
    cer_data = avg_by_model['cer'].dropna().sort_values()

    if len(wer_data) > 0 and len(cer_data) > 0:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

        colors_wer = ['green' if v == wer_data.min() else 'steelblue' for v in wer_data.values]
        ax1.barh(wer_data.index, wer_data.values, color=colors_wer)
        ax1.set_xlabel('Word Error Rate (WER)')
        ax1.set_title('WER per Model (Bigger Models)')
        for i, (model, v) in enumerate(wer_data.items()):
            ax1.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

        colors_cer = ['green' if v == cer_data.min() else 'coral' for v in cer_data.values]
        ax2.barh(cer_data.index, cer_data.values, color=colors_cer)
        ax2.set_xlabel('Character Error Rate (CER)')
        ax2.set_title('CER per Model (Bigger Models)')
        for i, (model, v) in enumerate(cer_data.items()):
            ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

        fig.suptitle('Bigger ASR Model Comparison (>1B Params)', fontsize=14, y=1.02)
        plt.tight_layout()

        chart_path = f'{BIGGER_EVAL_DIR}/evaluation_charts_bigger.png'
        plt.savefig(chart_path, dpi=150, bbox_inches='tight')
        print(f'\u2705 Chart saved: {chart_path}')
        plt.show()
    else:
        print('\u26A0  No metric data to plot.')


---
## Dialect-Level Soft Accuracy
A transcription is considered "correct" if WER < 0.3. Per-model, per-dialect accuracy.
Saved as `bigger_models/dialect_accuracy_bigger.csv`.


In [ ]:
WER_THRESHOLD = 0.3

if df.empty:
    print('\u26A0  No data for accuracy calculation.')
else:
    dialect_rows = []
    for model_name in df['model'].unique():
        model_df = df[df['model'] == model_name].copy()
        row = {'model': model_name}
        overall_correct = 0
        overall_total   = 0
        for dialect in DIALECT_FOLDERS:
            d_df = model_df[model_df['dialect'] == dialect]
            has_wer  = d_df['wer'].notna()
            correct  = (d_df.loc[has_wer, 'wer'] < WER_THRESHOLD).sum()
            total    = has_wer.sum()
            accuracy = round(correct / total, 3) if total > 0 else None
            row[dialect] = accuracy
            overall_correct += correct
            overall_total   += total
        row['OVERALL'] = round(overall_correct / overall_total, 3) if overall_total > 0 else None
        dialect_rows.append(row)

    dialect_df = pd.DataFrame(dialect_rows).set_index('model')
    dialect_acc_path = os.path.join(BIGGER_EVAL_DIR, 'dialect_accuracy_bigger.csv')
    dialect_df.to_csv(dialect_acc_path)
    print(f'\u2705 Saved: {dialect_acc_path}')

    display_cols = DIALECT_FOLDERS + ['OVERALL']
    print('\n=== Dialect-Level Soft Accuracy (WER < 0.3 = correct) ===')
    print(dialect_df[display_cols].to_string())


---
## Dialect-Level Confusion Matrix
Per-model correct/incorrect counts and WER heatmap per dialect.
Saved to `bigger_models/confusion_matrices/`.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

SOFT_THRESHOLD = 0.3

if df.empty:
    print('\u26A0  No data for confusion matrices.')
else:
    os.makedirs(f'{BIGGER_EVAL_DIR}/confusion_matrices', exist_ok=True)

    for model_name in df['model'].unique():
        model_df = df[df['model'] == model_name]
        valid = model_df[model_df['wer'].notna()]
        if len(valid) == 0:
            print(f'\u23ED  {model_name}: No WER data \u2014 skipping confusion matrix.')
            continue

        matrix_data = {}
        for dialect in DIALECT_FOLDERS:
            dialect_valid = valid[valid['dialect'] == dialect]
            correct   = (dialect_valid['wer'] < SOFT_THRESHOLD).sum()
            incorrect = len(dialect_valid) - correct
            matrix_data[dialect] = {'Correct (WER<0.3)': correct, 'Incorrect': incorrect}

        matrix_df = pd.DataFrame(matrix_data).T
        matrix_df.index.name = 'Dialect'

        dialect_metrics = {}
        for dialect in DIALECT_FOLDERS:
            d_valid = valid[valid['dialect'] == dialect]
            if len(d_valid) > 0:
                dialect_metrics[dialect] = {
                    'WER': d_valid['wer'].mean(),
                    'CER': d_valid['cer'].mean() if 'cer' in d_valid else None,
                }
            else:
                dialect_metrics[dialect] = {'WER': None, 'CER': None}

        metrics_df = pd.DataFrame(dialect_metrics).T

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(matrix_df, annot=True, fmt='d', cmap='RdYlGn', ax=ax1,
                    linewidths=0.5, cbar_kws={'label': 'Count'})
        ax1.set_title(f'{model_name} \u2014 Correct vs Incorrect per Dialect')
        ax1.set_ylabel('True Dialect')

        wer_matrix = metrics_df[['WER']].astype(float)
        sns.heatmap(wer_matrix, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax2,
                    linewidths=0.5, cbar_kws={'label': 'WER'}, vmin=0, vmax=1)
        ax2.set_title(f'{model_name} \u2014 WER per Dialect')
        ax2.set_ylabel('Dialect')

        plt.tight_layout()
        cm_path = f'{BIGGER_EVAL_DIR}/confusion_matrices/{model_name}_confusion.png'
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        print(f'\u2705 {model_name} confusion matrix saved: {cm_path}')
        plt.show()

    print(f'\n\u2705 All confusion matrices saved to: {BIGGER_EVAL_DIR}/confusion_matrices/')


---
## ✅ Complete

| Model | Params | Bengali | Status |
|---|---|---|---|
| qwen2_audio_direct | 7B | ⚠ Exp (8 langs) | Speech LLM Path A |
| qwen2_audio_via_english | 7B | ⚠ Exp | Speech LLM Path B |
| canary_1b | 1B | ⚠ Exp (4 langs) | FastConformer |
| phi4_multimodal | 5.6B | ⚠ Exp (8 langs) | Audio-LLM |
| qwen3_asr | 1.7B | ⚠ Exp (30 langs) | SOTA ASR (no Bengali) |
| mimo_audio | 7B | ⚠ Exp (zh/en) | Audio LLM |
